# Notebook 01 — Exploratory Data Analysis

**Project**: Diabetes Prediction using the Diabetes Health Indicators Dataset  
**Target**: `Diabetes_binary` (classes 0 = no diabetes, 1 = diabetes)

This notebook covers:
1. Imports and setup
2. Loading the dataset
3. Basic info (shape, dtypes, head)
4. Missing value analysis
5. Target variable distribution
6. Statistical summary
7. Feature distributions
8. Correlation heatmap
9. Box plots for outlier detection
10. Feature-target relationship plots
11. Key insights

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Add project root to Python path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

from src.data_loader import (
    load_dataset, basic_info, print_basic_info,
    validate_dataset, describe_dataset, get_feature_target_split
)
from src.visualization import (
    plot_distribution, plot_correlation_heatmap, plot_box_plots,
    plot_target_distribution, plot_feature_target_relationship
)
from src.config import RAW_DATASET_PATH, FEATURE_COLUMNS, TARGET_COLUMN, PLOTS_DIR

print('Libraries loaded successfully.')
print(f'Project root : {PROJECT_ROOT}')
print(f'Dataset path : {RAW_DATASET_PATH}')

Libraries loaded successfully.
Project root : /Users/tathagatadebnath/Downloads/federated_ML/v1-basic-ml
Dataset path : /Users/tathagatadebnath/Downloads/federated_ML/v1-basic-ml/data/raw/diabetes_binary.csv


## 2. Load Dataset

In [2]:
try:
    df = load_dataset(str(RAW_DATASET_PATH))
    print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
except FileNotFoundError as e:
    print(str(e))
    print()
    print('Creating a synthetic demo dataset to allow notebook execution...')
    rng = np.random.default_rng(42)
    n = 5000
    df = pd.DataFrame({
        'HighBP':              rng.integers(0, 2, n).astype(float),
        'HighChol':            rng.integers(0, 2, n).astype(float),
        'CholCheck':           rng.integers(0, 2, n).astype(float),
        'BMI':                 rng.normal(28, 6, n).clip(12, 80),
        'Smoker':              rng.integers(0, 2, n).astype(float),
        'Stroke':              rng.integers(0, 2, n).astype(float),
        'HeartDiseaseorAttack':rng.integers(0, 2, n).astype(float),
        'PhysActivity':        rng.integers(0, 2, n).astype(float),
        'Fruits':              rng.integers(0, 2, n).astype(float),
        'Veggies':             rng.integers(0, 2, n).astype(float),
        'HvyAlcoholConsump':   rng.integers(0, 2, n).astype(float),
        'AnyHealthcare':       rng.integers(0, 2, n).astype(float),
        'NoDocbcCost':         rng.integers(0, 2, n).astype(float),
        'GenHlth':             rng.integers(1, 6, n).astype(float),
        'MentHlth':            rng.integers(0, 31, n).astype(float),
        'PhysHlth':            rng.integers(0, 31, n).astype(float),
        'DiffWalk':            rng.integers(0, 2, n).astype(float),
        'Sex':                 rng.integers(0, 2, n).astype(float),
        'Age':                 rng.integers(1, 14, n).astype(float),
        'Education':           rng.integers(1, 7, n).astype(float),
        'Income':              rng.integers(1, 9, n).astype(float),
        'Diabetes_binary':     rng.choice([0, 1], n, p=[0.86, 0.14]).astype(float),
    })
    print(f'Synthetic dataset created: {df.shape}')

Dataset loaded: 253,680 rows x 22 columns


## 3. Basic Info

In [3]:
print('=== BASIC INFO ===')
print_basic_info(df)
print()
print('=== HEAD ===')
df.head()

=== BASIC INFO ===
Shape           : (253680, 22)
Duplicates      : 24206
Memory (MB)     : 42.579
Missing columns : None

Data types:
float64    22

=== HEAD ===


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [4]:
print('=== DATA TYPES ===')
print(df.dtypes)
print()
print('=== DATASET VALIDATION ===')
validation = validate_dataset(df)
for key, val in validation.items():
    print(f'  {key}: {val}')

=== DATA TYPES ===
Diabetes_binary         float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies                 float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
dtype: object

=== DATASET VALIDATION ===
  valid: True
  missing_columns: []
  unexpected_columns: []
  target_values: [0.0, 1.0]
  invalid_target_values: []
  fully_missing_columns: []
  warnings: []


## 4. Missing Value Analysis

In [5]:
info = basic_info(df)
missing_vals = info['missing_values']

if missing_vals:
    missing_series = pd.Series(missing_vals).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(missing_series.index, missing_series.values, color='coral')
    ax.set_title('Missing Values per Column', fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / 'missing_values.png'), bbox_inches='tight')
    plt.show()
else:
    print('No missing values found in the dataset.')

No missing values found in the dataset.


## 5. Target Variable Distribution

In [6]:
y = df[TARGET_COLUMN]
print('Class distribution:')
print(y.value_counts().sort_index())
print()
print('Class proportions:')
print(y.value_counts(normalize=True).sort_index().round(4))

plot_target_distribution(y, save_path=str(PLOTS_DIR / 'target_distribution.png'))

Class distribution:
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64

Class proportions:
Diabetes_binary
0.0    0.8607
1.0    0.1393
Name: proportion, dtype: float64


## 6. Statistical Summary

In [7]:
stats = describe_dataset(df)
print('Extended statistical summary (including skewness and kurtosis):')
stats

Extended statistical summary (including skewness and kurtosis):


,count,mean,std,min,25%,50%,75%,max,skewness,kurtosis
Diabetes_binary,253680.0,0.1393,0.3463,0.0,0.0,0.0,0.0,1.0,2.0830,2.3390
HighBP,253680.0,0.4290,0.4949,0.0,0.0,0.0,1.0,1.0,0.2869,-1.9177
HighChol,253680.0,0.4241,0.4942,0.0,0.0,0.0,1.0,1.0,0.3071,-1.9057
CholCheck,253680.0,0.9627,0.1896,0.0,1.0,1.0,1.0,1.0,-4.8813,21.8270
BMI,253680.0,28.3824,6.6087,12.0,24.0,27.0,31.0,98.0,2.1220,10.9975
Smoker,253680.0,0.4432,0.4968,0.0,0.0,0.0,1.0,1.0,0.2288,-1.9477
Stroke,253680.0,0.0406,0.1973,0.0,0.0,0.0,0.0,1.0,4.6573,19.6910
HeartDiseaseorAttack,253680.0,0.0942,0.2921,0.0,0.0,0.0,0.0,1.0,2.7787,5.7215
PhysActivity,253680.0,0.7565,0.4292,0.0,1.0,1.0,1.0,1.0,-1.1955,-0.5707
Fruits,253680.0,0.6343,0.4816,0.0,0.0,1.0,1.0,1.0,-0.5575,-1.6892


In [8]:
# Highlight highly skewed features
high_skew = stats[stats['skewness'].abs() > 1.0]['skewness'].sort_values(ascending=False)
print('Columns with |skewness| > 1:')
print(high_skew)

Columns with |skewness| > 1:
Stroke                  4.6573
HvyAlcoholConsump       3.8541
NoDocbcCost             2.9953
HeartDiseaseorAttack    2.7787
MentHlth                2.7211
PhysHlth                2.2074
BMI                     2.1220
Diabetes_binary         2.0830
DiffWalk                1.7739
PhysActivity           -1.1955
Veggies                -1.5922
AnyHealthcare          -4.1811
CholCheck              -4.8813
Name: skewness, dtype: float64


## 7. Feature Distributions

In [9]:
continuous_features = ['BMI', 'MentHlth', 'PhysHlth', 'GenHlth', 'Age', 'Education', 'Income']
plot_distribution(
    df, continuous_features,
    save_path=str(PLOTS_DIR / 'feature_distributions.png')
)

In [10]:
# Binary features
binary_features = [
    'HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke',
    'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
    'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex'
]
print('Binary feature value counts:')
for col in binary_features:
    if col in df.columns:
        counts = df[col].value_counts().sort_index()
        pct = (counts / len(df) * 100).round(1)
        print(f'  {col:<25} {dict(zip(counts.index, pct.values))} %')

Binary feature value counts:
  HighBP                    {0.0: np.float64(57.1), 1.0: np.float64(42.9)} %
  HighChol                  {0.0: np.float64(57.6), 1.0: np.float64(42.4)} %
  CholCheck                 {0.0: np.float64(3.7), 1.0: np.float64(96.3)} %
  Smoker                    {0.0: np.float64(55.7), 1.0: np.float64(44.3)} %
  Stroke                    {0.0: np.float64(95.9), 1.0: np.float64(4.1)} %
  HeartDiseaseorAttack      {0.0: np.float64(90.6), 1.0: np.float64(9.4)} %
  PhysActivity              {0.0: np.float64(24.3), 1.0: np.float64(75.7)} %
  Fruits                    {0.0: np.float64(36.6), 1.0: np.float64(63.4)} %
  Veggies                   {0.0: np.float64(18.9), 1.0: np.float64(81.1)} %
  HvyAlcoholConsump         {0.0: np.float64(94.4), 1.0: np.float64(5.6)} %
  AnyHealthcare             {0.0: np.float64(4.9), 1.0: np.float64(95.1)} %
  NoDocbcCost               {0.0: np.float64(91.6), 1.0: np.float64(8.4)} %
  DiffWalk                  {0.0: np.float64(83.2), 1

## 8. Correlation Heatmap

In [11]:
plot_correlation_heatmap(df, save_path=str(PLOTS_DIR / 'correlation_heatmap.png'))

In [12]:
# Top correlations with target
corr_with_target = df.corr()[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(ascending=False)
print('Top 10 features most correlated with target:')
print(corr_with_target.head(10).round(4))
print()
print('Bottom 5 (most negatively correlated):')
print(corr_with_target.tail(5).round(4))

Top 10 features most correlated with target:
GenHlth                 0.2936
HighBP                  0.2631
DiffWalk                0.2183
BMI                     0.2168
HighChol                0.2003
Age                     0.1774
HeartDiseaseorAttack    0.1773
PhysHlth                0.1713
Stroke                  0.1058
MentHlth                0.0693
Name: Diabetes_binary, dtype: float64

Bottom 5 (most negatively correlated):
Veggies             -0.0566
HvyAlcoholConsump   -0.0571
PhysActivity        -0.1181
Education           -0.1245
Income              -0.1639
Name: Diabetes_binary, dtype: float64


## 9. Box Plots for Outlier Detection

In [13]:
numeric_features = ['BMI', 'MentHlth', 'PhysHlth', 'GenHlth', 'Age', 'Education', 'Income']
plot_box_plots(
    df, numeric_features,
    save_path=str(PLOTS_DIR / 'box_plots.png')
)

In [14]:
# Quantify outliers per column using IQR
from src.preprocessor import detect_outliers_iqr

outlier_mask = detect_outliers_iqr(df, numeric_features)
print('Outlier counts per feature (IQR method, threshold=1.5):')
outlier_counts = outlier_mask.sum().sort_values(ascending=False)
print(outlier_counts)
print(f'\nTotal rows with at least one outlier: {outlier_mask.any(axis=1).sum():,}')

Outlier counts per feature (IQR method, threshold=1.5):
PhysHlth     40949
MentHlth     36208
GenHlth      12081
BMI           9847
Age              0
Education        0
Income           0
dtype: int64

Total rows with at least one outlier: 67,918


## 10. Feature-Target Relationship Plots

In [15]:
important_features = [
    'HighBP', 'HighChol', 'BMI', 'GenHlth', 'Age',
    'DiffWalk', 'HeartDiseaseorAttack', 'PhysActivity'
]
plot_feature_target_relationship(
    df, important_features, target_col=TARGET_COLUMN,
    save_path=str(PLOTS_DIR / 'feature_target_relationships.png')
)

In [16]:
# Cross-tabulations for key binary features
for feat in ['HighBP', 'HighChol', 'GenHlth']:
    if feat in df.columns:
        print(f'\n{feat} vs {TARGET_COLUMN} (normalised by row):')
        ct = pd.crosstab(df[feat], df[TARGET_COLUMN], normalize='index').round(3)
        print(ct)


HighBP vs Diabetes_binary (normalised by row):
Diabetes_binary    0.0    1.0
HighBP                       
0.0              0.940  0.060
1.0              0.756  0.244

HighChol vs Diabetes_binary (normalised by row):
Diabetes_binary   0.0   1.0
HighChol                   
0.0              0.92  0.08
1.0              0.78  0.22

GenHlth vs Diabetes_binary (normalised by row):
Diabetes_binary    0.0    1.0
GenHlth                      
1.0              0.975  0.025
2.0              0.928  0.072
3.0              0.822  0.178
4.0              0.690  0.310
5.0              0.621  0.379


## 11. Key Insights

### Dataset Summary
- **Size**: ~254k records (or synthetic demo data), 21 features + 1 target column.
- **Target**: `Diabetes_binary` — 2 classes: 0 (no diabetes), 1 (diabetes).
- **Class imbalance**: Class 0 dominates (~86%). SMOTE oversampling will be applied in preprocessing.

### Missing Values
- The raw dataset has no missing values (all features are survey responses coded as integers/floats).

### Outliers
- `BMI`, `MentHlth`, and `PhysHlth` exhibit right-skewed distributions with IQR outliers.
- Outlier capping (Winsorisation) will be applied rather than row removal to preserve data volume.

### Top Predictors
- **HighBP**, **HighChol**, **BMI**, **GenHlth**, and **Age** show the strongest correlations with the target.
- **PhysActivity** and **Fruits/Veggies** are negatively correlated with diabetes.

### Recommendations for Preprocessing
1. Cap outliers using IQR on continuous features (BMI, MentHlth, PhysHlth).
2. Apply StandardScaler — tree models benefit less but LR/NN require it.
3. Use SMOTE to address the class imbalance.
4. Stratified train-test split to preserve class proportions.

Next step: `02_preprocessing.ipynb`